<a href="https://colab.research.google.com/github/Arif0000/ML_BY_Arif/blob/main/Fraud_Detection_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install xgboost imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_path = "/content/drive/MyDrive/fraud/Data/"

In [ ]:
transactions = pd.read_csv(base_path + "Transaction Data/transaction_records.csv")
transaction_meta = pd.read_csv(base_path + "Transaction Data/transaction_metadata.csv")

customers = pd.read_csv(base_path + "Customer Profiles/customer_data.csv")
account_activity = pd.read_csv(base_path + "Customer Profiles/account_activity.csv")

fraud_indicators = pd.read_csv(base_path + "Fraudulent Patterns/fraud_indicators.csv")
suspicious = pd.read_csv(base_path + "Fraudulent Patterns/suspicious_activity.csv")

amount_data = pd.read_csv(base_path + "Transaction Amounts/amount_data.csv")
anomaly_scores = pd.read_csv(base_path + "Transaction Amounts/anomaly_scores.csv")

merchant_data = pd.read_csv(base_path + "Merchant Information/merchant_data.csv")
category_labels = pd.read_csv(base_path + "Merchant Information/transaction_category_labels.csv")

In [ ]:
df = transactions.copy()

# Merge step-by-step (adjust keys based on actual column names)
df = df.merge(transaction_meta, on="TransactionID", how="left")
df = df.merge(amount_data, on="TransactionID", how="left")
df = df.merge(anomaly_scores, on="TransactionID", how="left")

df = df.merge(customers, on="CustomerID", how="left")
df = df.merge(account_activity, on="CustomerID", how="left")

df = df.merge(merchant_data, on="MerchantID", how="left")
df = df.merge(category_labels, on="TransactionID", how="left")

# Target creation (important)
df = df.merge(fraud_indicators, on="TransactionID", how="left")

# Fill missing fraud label as 0
df['FraudIndicator'] = df['FraudIndicator'].fillna(0)

In [ ]:
df.head(100)

,TransactionID,Amount,CustomerID,Timestamp,MerchantID,TransactionAmount,AnomalyScore,Name,Age,Address,AccountBalance,LastLogin,MerchantName,Location,Category,FraudIndicator
0,1,55.530334,1952,2022-01-01 00:00:00,2701,79.413607,0.686699,Customer 1952,50,Address 1952,2869.689912,2024-08-09,Merchant 2701,Location 2701,Other,0
1,2,12.881180,1027,2022-01-01 01:00:00,2070,12.053087,0.081749,Customer 1027,46,Address 1027,9527.947107,2022-01-27,Merchant 2070,Location 2070,Online,0
2,3,50.176322,1955,2022-01-01 02:00:00,2238,33.310357,0.023857,Customer 1955,34,Address 1955,9288.355525,2024-08-12,Merchant 2238,Location 2238,Travel,0
3,4,41.634001,1796,2022-01-01 03:00:00,2879,46.121117,0.876994,Customer 1796,33,Address 1796,5588.049942,2024-03-06,Merchant 2879,Location 2879,Travel,0
4,5,78.122853,1946,2022-01-01 04:00:00,2966,54.051618,0.034059,Customer 1946,18,Address 1946,7324.785332,2024-08-03,Merchant 2966,Location 2966,Other,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,98.502686,1466,2022-01-04 23:00:00,2054,26.196352,0.495002,Customer 1466,41,Address 1466,2569.129234,2023-04-11,Merchant 2054,Location 2054,Retail,0
96,97,18.166845,1922,2022-01-05 00:00:00,2432,90.077827,0.304072,Customer 1922,46,Address 1922,5101.105372,2024-07-10,Merchant 2432,Location 2432,Other,0
97,98,55.887204,1183,2022-01-05 01:00:00,2346,21.111194,0.291575,Customer 1183,40,Address 1183,7149.964405,2022-07-02,Merchant 2346,Location 2346,Other,0
98,99,76.843429,1866,2022-01-05 02:00:00,2299,15.152685,0.241948,Customer 1866,35,Address 1866,9510.171444,2024-05-15,Merchant 2299,Location 2299,Online,0


In [ ]:
# Drop unnecessary columns
drop_cols = ['TransactionID', 'CustomerID']
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Handle missing values
df = df.fillna(df.median(numeric_only=True))

# Encode categorical columns
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
X = df.drop('FraudIndicator', axis=1)
y = df['FraudIndicator']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [ ]:
scaler = StandardScaler()
X_train_res = scaler.fit_transform(X_train_res)
X_test = scaler.transform(X_test)

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=10,
    eval_metric='logloss'
)

model.fit(X_train_res , y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       191
           1       0.00      0.00      0.00         9

    accuracy                           0.95       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.91      0.95      0.93       200

ROC-AUC: 0.38976148923792897


In [ ]:
threshold = 0.3
y_pred_custom = (y_prob > threshold).astype(int)

print("Custom Threshold:\n")
print(classification_report(y_test, y_pred_custom))

Custom Threshold:

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       191
           1       0.00      0.00      0.00         9

    accuracy                           0.95       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.91      0.95      0.93       200



In [ ]:
sample_batch = X_test[:10]

preds = model.predict(sample_batch)
probs = model.predict_proba(sample_batch)[:, 1]

for i in range(10):
    print(f"Transaction {i}: Fraud={preds[i]}, Score={probs[i]:.3f}")


Transaction 0: Fraud=0, Score=0.001
Transaction 1: Fraud=0, Score=0.001
Transaction 2: Fraud=0, Score=0.001
Transaction 3: Fraud=0, Score=0.001
Transaction 4: Fraud=0, Score=0.001
Transaction 5: Fraud=0, Score=0.001
Transaction 6: Fraud=0, Score=0.001
Transaction 7: Fraud=0, Score=0.001
Transaction 8: Fraud=0, Score=0.001
Transaction 9: Fraud=0, Score=0.001
